# Cuadernillo de prueba de concepto — LTI + nbgrader + envío al backend

Este cuadernillo junta todo lo que armamos hasta ahora en un solo archivo, para probar el flujo completo:

1. Lee los datos del estudiante que llegan por LTI (variables de entorno inyectadas por el `auth_state_hook`).
2. Dos ejercicios con verificación automática (tags compatibles con nbgrader) que capturan errores en vez de detener el cuadernillo.
3. Después de cada ejercicio, arma y muestra el `JSON` exacto que se enviaría al backend — **el backend todavía no existe, así que por ahora solo se imprime, no se envía**.
4. Al final, un resumen de cierre del intento.

> Este archivo es solo para probar el concepto — cuando el backend exista, cambiar `ENVIAR_AL_BACKEND` a `True` es lo único que hace falta para que empiece a mandar datos de verdad.

## 1. Datos del estudiante (vía LTI)

In [ ]:
import os
import json
import time
import uuid
import traceback
from datetime import datetime, timezone

VARIABLES_LTI = {
    "estudiante_id": "ALUMNO_ID",
    "nombre": "ALUMNO_NOMBRE",
    "correo": "ALUMNO_EMAIL",
    "rol": "ALUMNO_ROL",
    "curso_id": "CURSO_ID",
    "curso_nombre": "CURSO_NOMBRE",
}

datos_lti = {clave: os.environ.get(var, "") for clave, var in VARIABLES_LTI.items()}

if not any(datos_lti.values()):
    print("Sin datos de LTI en el entorno (¿corriste esto fuera de un lanzamiento desde Moodle?).")
    print("Se usarán valores de prueba para que el resto del cuadernillo funcione igual.")
    datos_lti = {
        "estudiante_id": "estudiante-prueba-001",
        "nombre": "Estudiante de prueba",
        "correo": "prueba@universidad.edu.co",
        "rol": "Learner",
        "curso_id": "curso-prueba",
        "curso_nombre": "Curso de prueba",
    }

for clave, valor in datos_lti.items():
    print(f"{clave}: {valor}")

## 2. Configuración de envío al backend (todavía no existe)

`enviar_evento` es la única función que habla con el backend. Ahora mismo, con `ENVIAR_AL_BACKEND = False`, solo imprime el JSON bonito — nada sale por red. El resto del cuadernillo llama siempre a esta función, así que activar el envío real más adelante es un cambio de una sola línea.

In [ ]:
BACKEND_URL = "https://tu-backend.example.com/api"
ENVIAR_AL_BACKEND = False  # cambia a True cuando el backend esté disponible
CUADERNILLO_ID = "poc-dos-ejercicios"


def enviar_evento(endpoint, payload):
    """
    Envía (o, mientras no haya backend, solo muestra) un evento.
    Retorna la respuesta del backend si ENVIAR_AL_BACKEND es True, o None si no.
    """
    print(f"--- POST {BACKEND_URL}{endpoint} ---")
    print(json.dumps(payload, indent=2, ensure_ascii=False))

    if not ENVIAR_AL_BACKEND:
        return None

    import requests
    respuesta = requests.post(f"{BACKEND_URL}{endpoint}", json=payload, timeout=5)
    respuesta.raise_for_status()
    return respuesta.json()

## 3. Inicio de la sesión (`intento_cuadernillo`)

Esto simula el `POST /intentos` que crea (o reutiliza) la fila en `intento_cuadernillo`. Como no hay backend todavía, el `intento_id` que normalmente vendría en la respuesta se genera aquí localmente, solo para que el resto del cuadernillo tenga algo con qué trabajar.

In [ ]:
payload_inicio = {
    "estudiante_id": datos_lti["estudiante_id"],
    "cuadernillo_id": CUADERNILLO_ID,
    "curso_id": datos_lti["curso_id"],
    "fecha_inicio": datetime.now(timezone.utc).isoformat(),
}

respuesta = enviar_evento("/intentos", payload_inicio)

# Cuando el backend exista, "intento_id" debería venir en la respuesta real:
# intento_id = respuesta["intento_id"]
# Por ahora lo simulamos:
intento_id = respuesta["intento_id"] if respuesta else str(uuid.uuid4())
print(f"\nintento_id: {intento_id}")

## 4. Verificación de ejercicios (captura errores e intentos)

In [ ]:
resultados_ejercicios = {}  # un registro por ejercicio_id, se acumula entre corridas


def verificar_ejercicio(id_ejercicio, funcion_prueba, puntos_maximos=1):
    inicio = time.time()
    previo = resultados_ejercicios.get(id_ejercicio, {"num_intentos": 0, "errores": []})

    resultado = {
        "ejercicio_id": id_ejercicio,
        "puntos_maximos": puntos_maximos,
        "puntos_obtenidos": 0,
        "aprobado": False,
        "num_intentos": previo["num_intentos"] + 1,
        "errores": list(previo["errores"]),
    }
    try:
        funcion_prueba()
        resultado["aprobado"] = True
        resultado["puntos_obtenidos"] = puntos_maximos
        print(f"[OK] {id_ejercicio}: correcto ({puntos_maximos} pts)")
    except AssertionError as e:
        resultado["errores"].append({
            "tipo": "AssertionError",
            "mensaje": str(e) or "El resultado no es el esperado",
            "fecha": datetime.now(timezone.utc).isoformat(),
        })
        print(f"[FALLO] {id_ejercicio}: el resultado no es el esperado")
    except Exception as e:
        resultado["errores"].append({
            "tipo": type(e).__name__,
            "mensaje": str(e),
            "fecha": datetime.now(timezone.utc).isoformat(),
        })
        print(f"[ERROR] {id_ejercicio}: tu código lanzó un error ({type(e).__name__})")

    resultado["duracion_segundos"] = round(time.time() - inicio, 2)
    resultados_ejercicios[id_ejercicio] = resultado
    return resultado


def enviar_resultado_ejercicio(id_ejercicio):
    resultado = resultados_ejercicios[id_ejercicio]
    payload = {"intento_id": intento_id, **resultado}
    enviar_evento("/resultados", payload)

## Ejercicio 1 — `es_par`

Completa la función `es_par(n)` para que retorne `True` si `n` es par y `False` si es impar.

In [ ]:
def es_par(n):
    ### BEGIN SOLUTION
    # return n % 2 == 0
    ### END SOLUTION
    raise NotImplementedError("Elimina esta línea e implementa la función")

In [ ]:
def _test_es_par():
    assert es_par(4) == True, "es_par(4) debería ser True"
    assert es_par(7) == False, "es_par(7) debería ser False"
    ### BEGIN HIDDEN TESTS
    assert es_par(0) == True, "es_par(0) debería ser True"
    assert es_par(-6) == True, "es_par(-6) debería ser True"
    ### END HIDDEN TESTS


verificar_ejercicio("ejercicio_1_es_par", _test_es_par, puntos_maximos=2)

JSON que se enviaría para este ejercicio:

In [ ]:
enviar_resultado_ejercicio("ejercicio_1_es_par")

## Ejercicio 2 — `suma_lista`

Completa la función `suma_lista(lista)` para que retorne la suma de todos los números de la lista.

In [ ]:
def suma_lista(lista):
    ### BEGIN SOLUTION
    # return sum(lista)
    ### END SOLUTION
    raise NotImplementedError("Elimina esta línea e implementa la función")

In [ ]:
def _test_suma_lista():
    assert suma_lista([1, 2, 3]) == 6
    ### BEGIN HIDDEN TESTS
    assert suma_lista([]) == 0, "la suma de una lista vacía debería ser 0"
    assert suma_lista([-1, 1]) == 0, "positivos y negativos deberían cancelarse"
    ### END HIDDEN TESTS


verificar_ejercicio("ejercicio_2_suma_lista", _test_suma_lista, puntos_maximos=3)

JSON que se enviaría para este ejercicio:

In [ ]:
enviar_resultado_ejercicio("ejercicio_2_suma_lista")

## 5. Resumen de cierre

Este resumen es opcional — el backend puede (y debería) decidir por su cuenta si el intento quedó `terminado`, comparando lo que ya le llegó de `resultado_ejercicio` contra el total de ejercicios del cuadernillo. Mandarlo desde aquí es solo un complemento útil, no la única fuente de verdad.

In [ ]:
puntaje_total = sum(r["puntos_obtenidos"] for r in resultados_ejercicios.values())
puntaje_maximo = sum(r["puntos_maximos"] for r in resultados_ejercicios.values())
todos_aprobados = all(r["aprobado"] for r in resultados_ejercicios.values()) and len(resultados_ejercicios) > 0

payload_resumen = {
    "intento_id": intento_id,
    "estudiante_id": datos_lti["estudiante_id"],
    "cuadernillo_id": CUADERNILLO_ID,
    "fecha_fin": datetime.now(timezone.utc).isoformat(),
    "estado": "terminado" if todos_aprobados else "en_progreso",
    "puntaje_total": puntaje_total,
    "puntaje_maximo": puntaje_maximo,
}

enviar_evento("/intentos/cerrar", payload_resumen)

if todos_aprobados:
    print("\nCompletaste los dos ejercicios del cuadernillo.")
else:
    pendientes = [eid for eid, r in resultados_ejercicios.items() if not r["aprobado"]]
    print(f"\nTe faltan por resolver: {', '.join(pendientes)}")